# 🏰 HTR Medieval French — Fine-tuning sur Kaggle

**Objectif :** Fine-tuner Kraken et TrOCR sur les manuscrits médiévaux français (CREMMA) pour atteindre un CER < 10%.

**Avantages Kaggle vs Colab :**
- Sessions de 12h (vs ~4h sur Colab gratuit)
- 30h GPU/semaine
- Pas de déconnexions aléatoires
- Output persistant dans `/kaggle/working/`

**⚠️ Important :** Activer le GPU → Settings → Accelerator → GPU T4 x2

## 1. Installation des dépendances

In [ ]:
# Vérifier le GPU
!nvidia-smi

In [ ]:
%%capture
# Installation des packages
!pip install kraken==5.3.0
!pip install transformers==4.40.0
!pip install peft==0.10.0
!pip install datasets==2.19.0
!pip install editdistance==0.8.1
!pip install Pillow>=10.0
!pip install scikit-learn>=1.5
!pip install accelerate>=0.26.0
!pip install sentencepiece
!pip install lxml

In [ ]:
import os
import json
import glob
import random
import numpy as np
import torch
from pathlib import Path
from PIL import Image
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

# Vérification GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Dossier de sortie persistant sur Kaggle
OUTPUT_BASE = "/kaggle/working/models"
os.makedirs(OUTPUT_BASE, exist_ok=True)
print(f"\n✅ Output persistant: {OUTPUT_BASE}")

In [ ]:
# Fixer les seeds pour reproductibilité
def fix_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

fix_seeds(42)

## 2. Téléchargement des données CREMMA Médiéval

In [ ]:
# Cloner CREMMA Médiéval
!git clone https://github.com/HTR-United/cremma-medieval data/cremma
print("\n✅ CREMMA Médiéval cloné")

In [ ]:
# Explorer la structure
cremma_dir = Path("data/cremma/data")
manuscripts = sorted([d.name for d in cremma_dir.iterdir() if d.is_dir()])
print(f"Nombre de manuscrits : {len(manuscripts)}")
for ms in manuscripts:
    n_xml = len([f for f in (cremma_dir / ms).glob("*.xml") if "chocomufin" not in f.name])
    n_jpg = len(list((cremma_dir / ms).glob("*.jpg")))
    print(f"  {ms}: {n_jpg} images, {n_xml} XML")

## 3. Extraction des lignes depuis ALTO XML

In [ ]:
import xml.etree.ElementTree as ET

def extract_lines_from_alto(xml_path, img_path):
    """
    Extrait les lignes d'un fichier ALTO XML.
    Retourne une liste de dicts {'image': PIL.Image, 'text': str, 'line_id': str}
    """
    records = []
    
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
    except ET.ParseError:
        return records
    
    ns = ""
    if root.tag.startswith("{"):
        ns = root.tag.split("}")[0] + "}"
    
    try:
        page_img = Image.open(img_path).convert("RGB")
    except Exception as e:
        print(f"  [SKIP] Cannot open {img_path}: {e}")
        return records
    
    for i, text_line in enumerate(root.iter(f"{ns}TextLine")):
        parts = []
        for string_el in text_line.iter(f"{ns}String"):
            content = string_el.get("CONTENT", "")
            if content:
                parts.append(content)
        
        text = " ".join(parts).strip()
        if not text or len(text) < 2:
            continue
        
        try:
            hpos = float(text_line.get("HPOS", 0))
            vpos = float(text_line.get("VPOS", 0))
            width = float(text_line.get("WIDTH", 0))
            height = float(text_line.get("HEIGHT", 0))
        except (ValueError, TypeError):
            continue
        
        if width <= 0 or height <= 0:
            continue
        
        x0 = max(0, int(hpos))
        y0 = max(0, int(vpos))
        x1 = min(page_img.width, int(hpos + width))
        y1 = min(page_img.height, int(vpos + height))
        
        if x1 <= x0 or y1 <= y0:
            continue
        
        line_img = page_img.crop((x0, y0, x1, y1))
        
        line_id = f"{xml_path.stem}_l{i:03d}"
        records.append({
            "image": line_img,
            "text": text,
            "line_id": line_id,
        })
    
    return records

In [ ]:
# Extraire toutes les lignes — sauvegarde sur disque
lines_dir = Path("data/lines")
lines_dir.mkdir(parents=True, exist_ok=True)

all_records = []
cremma_dir = Path("data/cremma/data")
line_counter = 0

for ms_dir in sorted(cremma_dir.iterdir()):
    if not ms_dir.is_dir():
        continue
    
    ms_name = ms_dir.name
    xml_files = sorted([
        f for f in ms_dir.glob("*.xml")
        if "chocomufin" not in f.name
    ])
    
    ms_count = 0
    for xml_path in xml_files:
        img_path = xml_path.with_suffix(".jpg")
        if not img_path.exists():
            continue
        
        records = extract_lines_from_alto(xml_path, img_path)
        for r in records:
            line_img_path = lines_dir / f"{line_counter:05d}.png"
            r["image"].save(line_img_path)
            
            all_records.append({
                "img_path": str(line_img_path),
                "text": r["text"],
                "line_id": r["line_id"],
                "manuscript": ms_name,
            })
            line_counter += 1
        ms_count += len(records)
    
    print(f"  {ms_name}: {ms_count} lignes")

print(f"\n✅ Total: {len(all_records)} lignes extraites")

## 4. Split Train/Val par manuscrit

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

manuscripts_list = [r["manuscript"] for r in all_records]

gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_idx, val_idx = next(gss.split(all_records, groups=manuscripts_list))

train_records = [all_records[i] for i in train_idx]
val_records = [all_records[i] for i in val_idx]

train_ms = set(r["manuscript"] for r in train_records)
val_ms = set(r["manuscript"] for r in val_records)

print(f"Train: {len(train_records)} lignes ({len(train_ms)} manuscrits)")
print(f"Val:   {len(val_records)} lignes ({len(val_ms)} manuscrits)")
assert len(train_ms & val_ms) == 0, "Data leakage!"
print("\n✅ Pas de data leakage")

---
## 5. Fine-tuning Kraken (CNN+LSTM)

In [ ]:
# Préparer les données pour ketos (format path: image + .gt.txt)
kraken_train_dir = Path("data/kraken_train")
kraken_val_dir = Path("data/kraken_val")
kraken_train_dir.mkdir(parents=True, exist_ok=True)
kraken_val_dir.mkdir(parents=True, exist_ok=True)

def save_kraken_format(records, output_dir):
    manifest = []
    for i, record in enumerate(tqdm(records, desc=f"Saving to {output_dir.name}")):
        img_path = output_dir / f"line_{i:05d}.png"
        txt_path = output_dir / f"line_{i:05d}.gt.txt"
        
        img = Image.open(record["img_path"]).convert("L")
        img.save(img_path)
        
        with open(txt_path, "w", encoding="utf-8") as f:
            f.write(record["text"])
        
        manifest.append(str(img_path))
    return manifest

train_manifest = save_kraken_format(train_records, kraken_train_dir)
val_manifest = save_kraken_format(val_records, kraken_val_dir)

# Sauvegarder les manifests
with open("train_manifest.txt", "w") as f:
    for path in train_manifest:
        f.write(path + "\n")

with open("val_manifest.txt", "w") as f:
    for path in val_manifest:
        f.write(path + "\n")

print(f"\n✅ Train: {len(train_manifest)} fichiers")
print(f"✅ Val:   {len(val_manifest)} fichiers")

In [ ]:
%%time
import os
os.environ["PATH"] += ":/root/.local/bin:/usr/local/bin:/opt/conda/bin"

# Vérifier que ketos est accessible
!which ketos

# Fine-tuning Kraken
!ketos train \
    -f path \
    -d cuda:0 \
    --augment \
    --workers 2 \
    --lag 10 \
    --min-epochs 5 \
    --epochs 50 \
    -o {OUTPUT_BASE}/kraken_cremma_finetuned \
    --training-files train_manifest.txt \
    --evaluation-files val_manifest.txt

In [ ]:
# Trouver le meilleur modèle Kraken
kraken_models = sorted(glob.glob(f"{OUTPUT_BASE}/kraken_cremma_finetuned*.mlmodel"))
if kraken_models:
    best_kraken = kraken_models[-1]
    print(f"✅ Meilleur modèle Kraken: {best_kraken}")
else:
    print("⚠️ Aucun modèle Kraken trouvé")
    # Chercher partout
    all_ml = glob.glob("**/*.mlmodel", recursive=True)
    print(f"   Tous les .mlmodel: {all_ml}")

---
## 6. Fine-tuning TrOCR avec LoRA

In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, EarlyStoppingCallback
from peft import LoraConfig, get_peft_model
import editdistance

print("✅ Imports OK")

In [ ]:
# Charger le modèle de base
MODEL_NAME = "microsoft/trocr-base-handwritten"

print(f"Chargement de {MODEL_NAME}...")
processor = TrOCRProcessor.from_pretrained(MODEL_NAME)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)
print(f"Paramètres totaux: {model.num_parameters():,}")

In [ ]:
# Appliquer LoRA r=8
LORA_R = 8

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_R * 4,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Config decoder
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

model = model.to(device)
print(f"Modèle sur {device}")

In [ ]:
# Dataset lazy-loading
MAX_LENGTH = 128

class LazyHTRDataset(torch.utils.data.Dataset):
    def __init__(self, records, processor, max_length=MAX_LENGTH):
        self.processor = processor
        self.max_length = max_length
        self.img_paths = [r["img_path"] for r in records]
        self.texts = [r["text"] for r in records]
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        img = Image.open(self.img_paths[idx]).convert("RGB")
        pixel_values = self.processor(img, return_tensors="pt").pixel_values[0]
        
        labels = self.processor.tokenizer(
            self.texts[idx],
            padding="max_length",
            max_length=self.max_length,
            truncation=True,
            return_tensors="pt",
        ).input_ids[0]
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        
        return {"pixel_values": pixel_values, "labels": labels}

train_dataset = LazyHTRDataset(train_records, processor)
val_dataset = LazyHTRDataset(val_records, processor)

print(f"✅ Train: {len(train_dataset)} samples")
print(f"✅ Val:   {len(val_dataset)} samples")

In [ ]:
# Métriques
def compute_cer(predictions, references):
    total_errors = sum(editdistance.eval(p, r) for p, r in zip(predictions, references))
    total_chars = sum(len(r) for r in references)
    return total_errors / total_chars if total_chars > 0 else 0.0

def compute_metrics(pred):
    labels_ids = pred.label_ids
    pred_ids = pred.predictions
    labels_ids[labels_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(labels_ids, skip_special_tokens=True)
    cer = compute_cer(pred_str, label_str)
    return {"cer": cer}

In [ ]:
# Configuration
EPOCHS = 30
BATCH_SIZE = 8
LEARNING_RATE = 5e-5
TROCR_OUTPUT_DIR = f"{OUTPUT_BASE}/trocr-cremma-lora"

training_args = Seq2SeqTrainingArguments(
    output_dir=TROCR_OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    predict_with_generate=True,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,
    fp16=True,
    seed=42,
    logging_steps=25,
    logging_dir=f"{TROCR_OUTPUT_DIR}/logs",
    report_to="none",
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    save_total_limit=3,
    dataloader_num_workers=2,
)

print(f"Epochs: {EPOCHS}, Batch: {BATCH_SIZE}, LR: {LEARNING_RATE}, LoRA r={LORA_R}")

In [ ]:
# Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
)

print("✅ Trainer prêt")

In [ ]:
%%time
# 🚀 LANCER LE FINE-TUNING (reprend si un checkpoint existe)
print("="*60)
print("🚀 Fine-tuning TrOCR + LoRA r=8")
print("="*60)

checkpoints = sorted(glob.glob(f"{TROCR_OUTPUT_DIR}/checkpoint-*"))
if checkpoints:
    print(f"📂 Reprise depuis: {checkpoints[-1]}")
    train_result = trainer.train(resume_from_checkpoint=checkpoints[-1])
else:
    print("🆕 Entraînement from scratch")
    train_result = trainer.train()

print(f"\n✅ Terminé! Loss: {train_result.training_loss:.4f}")

In [ ]:
# Sauvegarder le modèle final
trainer.save_model(TROCR_OUTPUT_DIR)
processor.save_pretrained(TROCR_OUTPUT_DIR)
print(f"✅ Modèle sauvegardé: {TROCR_OUTPUT_DIR}")

In [ ]:
# Évaluation finale
eval_results = trainer.evaluate()

print("\n" + "="*60)
print("📊 RÉSULTATS TrOCR + LoRA r=8")
print("="*60)
print(f"CER: {eval_results['eval_cer']:.4f} ({eval_results['eval_cer']*100:.1f}%)")
print(f"Loss: {eval_results['eval_loss']:.4f}")
print("="*60)

if eval_results['eval_cer'] < 0.10:
    print("\n🎉 CER < 10% — Objectif atteint!")
else:
    print(f"\n📈 CER = {eval_results['eval_cer']*100:.1f}% — à améliorer")

## 7. Évaluation qualitative

In [ ]:
# Prédictions sur des exemples de validation
model.eval()
n_examples = 10
sample_indices = random.sample(range(len(val_records)), min(n_examples, len(val_records)))

preds_sample = []
refs_sample = []

for idx in sample_indices:
    record = val_records[idx]
    img = Image.open(record["img_path"]).convert("RGB")
    
    pixel_values = processor(img, return_tensors="pt").pixel_values.to(device)
    with torch.no_grad():
        generated_ids = model.generate(pixel_values, max_new_tokens=128)
    
    pred_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    ref_text = record["text"]
    
    preds_sample.append(pred_text)
    refs_sample.append(ref_text)
    
    line_cer = editdistance.eval(pred_text, ref_text) / max(len(ref_text), 1)
    marker = "✓" if line_cer < 0.10 else "✗"
    print(f"{marker} REF:  {ref_text[:70]}")
    print(f"  PRED: {pred_text[:70]}")
    print(f"  CER:  {line_cer:.1%}\n")

print(f"CER échantillon: {compute_cer(preds_sample, refs_sample):.1%}")

## 8. Résumé final

In [ ]:
print("\n" + "="*60)
print("📊 RÉSUMÉ DES MODÈLES")
print("="*60)

# Kraken
kraken_models = sorted(glob.glob(f"{OUTPUT_BASE}/kraken_cremma_finetuned*.mlmodel"))
if kraken_models:
    print(f"\n1. Kraken fine-tuné: {kraken_models[-1]}")
else:
    print("\n1. Kraken: entraînement en cours ou échoué")

# TrOCR
print(f"2. TrOCR LoRA r=8: {TROCR_OUTPUT_DIR}")
if 'eval_results' in dir():
    print(f"   CER = {eval_results['eval_cer']*100:.1f}%")

print(f"\n💾 Tous les modèles dans: {OUTPUT_BASE}")
print("   → Télécharger depuis l'onglet 'Output' de Kaggle")